# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading, exploration, and analysis of the [FAIR²](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` Python library. All entities—record sets, fields, and columns—are referenced by their `@id`.

## Dataset Source
The dataset source is described by the Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and explore with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print name and description from dataset.metadata attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their `@id`, then inspect the fields of each record set.

In [ ]:
# This will list the available record sets in the dataset schema

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in the dataset schema.")
else:
    print(f"Number of record sets: {len(record_sets)}\n")
    for idx, rs in enumerate(record_sets):
        print(f"{idx + 1}. @id: {rs['@id']}")
        print(f"   Name: {rs.get('name', '-')}")
        print(f"   Description: {rs.get('description', '-')}")

    # Show fields (columns) for each record set
    for rs in record_sets:
        print(f"\nFields for record set '@id': {rs['@id']}")
        if 'field' in rs and rs['field']:
            for f in rs['field']:
                # Each field is either dict or reference
                if isinstance(f, dict):
                    print(f"  - {f.get('@id', '[no @id]')}: {f.get('name', '')}")
                else:
                    print(f"  - {f}")
        else:
            print("  No fields listed.")

# For demonstration, show up to the first few records from each record set by @id using mlcroissant
if record_sets:
    print("\nPreviewing first 2 records for each record set:")
    for rs in record_sets:
        print(f"\nRecord set @id: {rs['@id']}")
        try:
            for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
                print(f"Record {i+1}: {rec}")
                if i >= 1:  # print only first 2 records
                    break
        except Exception as e:
            print(f"  Could not read records for record set: {rs['@id']} | Error: {str(e)}")

## 3. Data Extraction
Extract all records from each record set into Pandas DataFrames for analysis. The record set and field `@id` values are used for referencing throughout.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else []
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set @id: {record_set_id} with shape {df.shape}")
        else:
            print(f"Record set @id: {record_set_id} contains no records.")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id} | Error: {str(e)}")

if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nColumns in first record set (@id: {first_id}):\n{dataframes[first_id].columns.tolist()}")
    print(dataframes[first_id].head())
else:
    print("No dataframes loaded. Perhaps the dataset has empty or undefined record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply EDA to a numeric field in a record set. First, we select a record set and then pick a numeric field by its `@id` for demonstration. The example includes filtering and normalization.

In [ ]:
# Let's demonstrate on the first available valid record set with numeric data
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dataframes.items():
    # Attempt to automatically guess numeric columns
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        selected_record_set_id = rs_id
        numeric_field_id = num_cols[0]  # pick first numeric field
        # Try to find a potential group field (string column with <20 unique vals)
        group_cands = [c for c in df.columns if (df[c].dtype == object and df[c].nunique() < 20)]
        if group_cands:
            group_field_id = group_cands[0]
        break

if selected_record_set_id is None:
    print("No suitable record set with numeric field found for EDA.")
else:
    print(f"Demonstrating EDA for record set @id: {selected_record_set_id}, numeric field: {numeric_field_id}")
    df = dataframes[selected_record_set_id]
    # Remove missing values for the analysis
    filtered_df = df[df[numeric_field_id].notnull()]  # only non-null
    threshold = filtered_df[numeric_field_id].mean()  # Example: use mean as threshold
    subset_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(subset_df.head())

    # Normalize the numeric field (z-score)
    subset_df[f"{numeric_field_id}_normalized"] = (subset_df[numeric_field_id] - subset_df[numeric_field_id].mean()) / subset_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(subset_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally, group by a categorical field, if available
    if group_field_id is not None:
        grouped_df = subset_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and, if applicable, its grouping across categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field_id is not None:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    df[selected_record_set_id] = df  # workaround for diagnostic reference
    sns.histplot(df[numeric_field_id].dropna(), bins=20, ax=ax[0], kde=True, color='steelblue')
    ax[0].set_title(f'Distribution of {numeric_field_id}')

    if group_field_id is not None:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=ax[1])
        ax[1].set_title(f'{numeric_field_id} by {group_field_id}')
    else:
        sns.kdeplot(df[numeric_field_id].dropna(), fill=True, ax=ax[1], color='coral')
        ax[1].set_title(f'KDE of {numeric_field_id}')

    plt.tight_layout()
    plt.show()

## 6. Conclusion

* In this notebook we demonstrated how to explore the FAIR² dataset via its Croissant schema with `mlcroissant`.
* Record sets, fields, and columns are referenced exclusively by their `@id`, ensuring unambiguous referencing and reproducibility.
* Initial EDA and visualization help reveal distributional patterns in the data that may inform further modeling or domain-driven analysis.

_For more details and dataset documentation, see the [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)._